# 02 - Cleaning & Target Definition

**Influenza Season Forecasting** - Notebook 2 of 6

**Purpose:** From the verified ILINet series, produce a per-season cleaned table and the two
prediction targets (`peak_ili_pct`, `peak_week`), and build the stitched NREVSS
`dominant_strain` series across the 2015-16 reporting break.

**This notebook hardens methodological choices (target definitions). Per CLAUDE.md it stops
for review before anything is committed.** (Reviewed and approved across four rounds: season
alignment, strain seam, the wk52 holiday artifact, and the flag definitions below.)

Scope notes (still pending Dr. Mitra, treated as assumptions):
- National only, `% WEIGHTED ILI` as the ILI target.
- Season = MMWR week 40 through week 39 of the next calendar year.
- Pandemic seasons (2009-10 H1N1, 2020-21) kept and labeled, not dropped. 2008-09 is labeled
  *pandemic-adjacent (2009 H1N1 emergence)*, traceably distinct from 2009-10.

**Targets are defined on a 3-week centered moving average of `% WEIGHTED ILI`** to remove the
year-end (wk52) holiday reporting artifact (see section 3). Raw values are retained alongside.

**Leakage wall (explicit).** The centered smoother used here looks at the week *after* the peak.
That is legitimate for a TARGET, which is defined retrospectively over a completed season. It has
nothing to do with the decision-week cutoff that governs FEATURES in `05_forecasting.ipynb`, where
inputs are sliced to data through week W. Retrospective target smoothing and decision-time feature
windows are two different things and are kept separate.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = next((Path(p) for p in ["data/raw", "../data/raw"] if Path(p).exists()), Path("data/raw"))
ILINET = "ILINet.csv"
NREVSS_COMBINED = "ICL_NREVSS_Combined_prior_to_2015_16.csv"   # pre-2015-16, has subtype
NREVSS_PHL      = "ICL_NREVSS_Public_Health_Labs.csv"          # 2015-16 on, has subtype
# Clinical Labs is positivity-only and is intentionally NOT used for subtype.

PANDEMIC_SEASONS = ["2009-10", "2020-21"]
PANDEMIC_ADJACENT = {"2008-09": "pandemic-adjacent (2009 H1N1 emergence)"}
SMOOTH_WIN = 3          # default centered window for target definition
HOLIDAY_WEEKS = {51, 52, 1}
pd.set_option("display.width", 140)
print("data dir:", DATA_DIR.resolve())

## 1. Season alignment

**Convention.** A flu season runs from **MMWR week 40 of the start year through week 39 of the
next calendar year**. The label is `"<startYY>-<endYY>"`.

- Weeks **40-52 (or 53)** belong to the season that **starts that calendar year**.
- Weeks **1-39** belong to the season that **started the prior calendar year**.

Example: `"2018-19"` = week 40 of 2018 through week 39 of 2019. The mapping is a deterministic
function of `(YEAR, WEEK)`, so no `(YEAR, WEEK)` pair can fall in two seasons; we assert that.

In [ ]:
def season_of(year, week):
    """Map an MMWR (year, week) to its flu season.
    Returns (label, start_year). Week >= 40 starts a season; weeks 1-39 belong to
    the season that started the previous calendar year."""
    start_year = year if week >= 40 else year - 1
    label = f"{start_year}-{str(start_year + 1)[2:]}"
    return label, start_year

print("Season convention: '2018-19' = MMWR wk40 2018 -> wk39 2019")
print("  weeks 40-52/53 -> season starting that year; weeks 1-39 -> season that started prior year")

In [ ]:
ili = pd.read_csv(DATA_DIR / ILINET, skiprows=1, na_values=["X"])
_info = ili.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
ili["season"] = [x[0] for x in _info]
ili["season_start_year"] = [x[1] for x in _info]
# Within-season ordering key: weeks 40..53 (0xx) then weeks 1..39 (1xx).
ili["season_order"] = ili["WEEK"].apply(lambda w: w if w >= 40 else w + 100)

# Assertion A: no (YEAR, WEEK) is assigned to more than one season.
assert ili.groupby(["YEAR", "WEEK"])["season"].nunique().eq(1).all(), "a (YEAR,WEEK) mapped to >1 season"
# Assertion B: ILINet has no duplicate (YEAR, WEEK) rows.
assert not ili.duplicated(["YEAR", "WEEK"]).any(), "duplicate (YEAR,WEEK) rows in ILINet"

print("rows:", len(ili), "| seasons:", ili["season"].nunique())
print("assertions passed: single-valued season map, no duplicate weeks")

### Contiguity and completeness

For each season we split rows into the start-year part (weeks 40..52/53) and the next-year part
(weeks 1..39), and check each part is gap-free, the season starts at week 40, and ends at week 39.
A season missing either boundary (the 2003 start, or an in-progress current tail) is flagged
**incomplete** and excluded from target construction.

In [ ]:
def _consecutive(weeks):
    weeks = sorted(weeks)
    return all(b - a == 1 for a, b in zip(weeks, weeks[1:])) if len(weeks) > 1 else True

rows = []
for s, g in ili.groupby("season"):
    sy = int(g["season_start_year"].iloc[0])
    start_part = sorted(g.loc[g["YEAR"] == sy, "WEEK"].tolist())
    end_part   = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"].tolist())
    complete = (len(start_part) > 0 and start_part[0] == 40 and
                len(end_part) > 0 and end_part[0] == 1 and end_part[-1] == 39)
    contiguous = _consecutive(start_part) and _consecutive(end_part)
    rows.append({"season": s, "start_year": sy, "n_weeks": len(g),
                 "first_wk": (start_part or [None])[0], "last_wk": (end_part or [None])[-1],
                 "complete": complete, "contiguous": contiguous})

coverage = pd.DataFrame(rows).sort_values("start_year").reset_index(drop=True)
assert coverage["contiguous"].all(), "a season has non-contiguous weeks"  # Assertion C
incomplete = coverage.loc[~coverage["complete"], "season"].tolist()
print(coverage.to_string(index=False))
print("\nincomplete seasons (excluded from targets):", incomplete if incomplete else "none")

## 2. Targets (defined on a 3-week centered smoother)

For each complete season we compute, on the **3-week centered moving average** of `% WEIGHTED ILI`
in season order:
- `peak_ili_pct` = smoothed maximum
- `peak_week`    = MMWR week of that smoothed maximum

and retain the unsmoothed values as `peak_ili_pct_raw` / `peak_week_raw`. The smoother removes the
single-week year-end holiday spike documented in section 3. A 5-week smoothed peak week is also
computed, only to detect fragile (smoothing-window-sensitive) labels.

Tie handling on the raw series uses first occurrence in season order; `n_tied_max` records ties.

In [ ]:
def peak_on_smoothed(g, win):
    """Return (peak_week, peak_value) of the centered-`win` moving average over season order."""
    g = g.sort_values("season_order")
    sm = g["% WEIGHTED ILI"].rolling(win, center=True).mean()
    i = sm.idxmax()
    return int(g.loc[i, "WEEK"]), round(float(sm.loc[i]), 3)

complete_seasons = coverage.loc[coverage["complete"], "season"].tolist()
comp = ili[ili["season"].isin(complete_seasons)].copy()

targets = []
for s, g in comp.groupby("season"):
    g = g.sort_values("season_order")
    # raw
    raw_max = g["% WEIGHTED ILI"].max()
    n_tied = int((g["% WEIGHTED ILI"] == raw_max).sum())
    raw_week = int(g[g["% WEIGHTED ILI"] == raw_max].iloc[0]["WEEK"])
    # smoothed (default 3) and 5 for fragility test
    pk3_week, pk3_val = peak_on_smoothed(g, SMOOTH_WIN)
    pk5_week, _ = peak_on_smoothed(g, 5)
    targets.append({"season": s,
                    "peak_week": pk3_week, "peak_ili_pct": pk3_val,
                    "peak_week_raw": raw_week, "peak_ili_pct_raw": round(float(raw_max), 3),
                    "peak_week_sm5": pk5_week, "n_tied_max": n_tied})

targets = pd.DataFrame(targets).sort_values("season").reset_index(drop=True)
print("complete seasons with targets:", len(targets))
print("tied raw maxima:", targets.loc[targets["n_tied_max"] > 1, "season"].tolist() or "none")

### Flag columns

Three flags, each literally true to its name:

- **`holiday_shift`** = raw peak week in {51, 52, 1} **and** smoothed peak week != raw. The wk52
  holiday artifact actually moved the label. Expected 5: 2003-04, 2005-06, 2012-13, 2013-14, 2019-20.
- **`peak_week_smoothing_sensitive`** = smoothed peak week != raw (any week). Broader marker that
  the label moved at all under smoothing; *not* a holiday claim. Kept distinct from `holiday_shift`.
- **`fragile_peak_week`** = 3-week and 5-week smoothed peak weeks disagree. The peak timing is
  genuinely uncertain to within ~1 week (flat plateau seasons). General criterion, not special-cased.

Special-case labels (`is_pandemic`, `special_case`) mark 2009-10 / 2020-21 (pandemic) and 2008-09
(pandemic-adjacent), kept in the output, not dropped.

In [ ]:
t = targets
t["holiday_shift"] = t["peak_week_raw"].isin(HOLIDAY_WEEKS) & (t["peak_week"] != t["peak_week_raw"])
t["peak_week_smoothing_sensitive"] = t["peak_week"] != t["peak_week_raw"]
t["fragile_peak_week"] = t["peak_week"] != t["peak_week_sm5"]
t["is_pandemic"] = t["season"].isin(PANDEMIC_SEASONS)
t["special_case"] = t["season"].map(lambda s: "pandemic" if s in PANDEMIC_SEASONS
                                     else PANDEMIC_ADJACENT.get(s, ""))

print("holiday_shift (5 expected):", t.loc[t["holiday_shift"], "season"].tolist())
print("peak_week_smoothing_sensitive:", t.loc[t["peak_week_smoothing_sensitive"], "season"].tolist())
print("fragile_peak_week:", t.loc[t["fragile_peak_week"], "season"].tolist())
print("special cases:", {r.season: r.special_case for r in t.itertuples() if r.special_case})

assert t["holiday_shift"].sum() == 5, "holiday_shift set changed unexpectedly"

### Full per-season target table

In [ ]:
cols = ["season", "peak_week", "peak_ili_pct", "peak_week_raw", "peak_ili_pct_raw",
        "peak_week_sm5", "holiday_shift", "peak_week_smoothing_sensitive",
        "fragile_peak_week", "special_case"]
print(targets[cols].to_string(index=False))

## 3. The wk52 holiday artifact (the reason targets are smoothed)

Diagnostic across **all 22 seasons**: raw vs 3-week-smoothed peak week, and how far raw wk52 sits
above its own local 3-week mean. The artifact signature is a large shift confined to wk52-peaking
seasons. If any *non*-wk52 season showed a large (>2 week) shift, that would be a different artifact
needing its own handling. The check below asserts none does.

In [ ]:
diag = []
for s, g in comp.groupby("season"):
    g = g.sort_values("season_order").reset_index(drop=True)
    sm = g["% WEIGHTED ILI"].rolling(SMOOTH_WIN, center=True).mean()
    raw_week = int(g.loc[g["% WEIGHTED ILI"].idxmax(), "WEEK"])
    sm_week = int(g.loc[sm.idxmax(), "WEEK"])
    shift = abs((sm_week if sm_week >= 40 else sm_week + 100) -
                (raw_week if raw_week >= 40 else raw_week + 100))
    spike = None
    if 52 in set(g["WEEK"]):
        j = g.index[g["WEEK"] == 52][0]
        spike = round(float(g.loc[j, "% WEIGHTED ILI"] - sm.loc[j]), 2)
    diag.append({"season": s, "raw_pk": raw_week, "sm3_pk": sm_week,
                 "shift_wks": shift, "raw_pk_wk52": raw_week == 52, "wk52_spike": spike})
diag = pd.DataFrame(diag).sort_values("season").reset_index(drop=True)
print(diag.to_string(index=False))

big = diag[diag["shift_wks"] > 2]
non_wk52_big = big[~big["raw_pk_wk52"]]
print("\nseasons with large (>2 wk) raw->smoothed shift:", big["season"].tolist())
print("of those, NOT peaking at wk52 (would be a new artifact):", non_wk52_big["season"].tolist() or "none")
assert non_wk52_big.empty, "a non-wk52 large shift appeared: investigate before trusting smoothing"

## 4. NREVSS strain stitch -> `dominant_strain`

Subtype counts bucketed into three circulating strains; dominant = bucket with most specimens:

- **A(H1N1)** = `A (H1)` + `A (2009 H1N1)`
- **A(H3N2)** = `A (H3)`
- **B** = `B` + `BVic` + `BYam`

Excluded from the vote: `A (Subtyping not Performed)`, `A (Unable to Subtype)` (not a strain),
`H3N2v`, `A (H5)` (sporadic zoonotic). **Stitch:** Combined for `season_start_year <= 2014`,
Public Health Labs for `>= 2015`. Clinical Labs is not used. Validated against the historical
record at the seam: 2014-15 A(H3N2), 2015-16 A(H1N1), 2016-17 A(H3N2).

In [ ]:
def load_nrevss(fname):
    d = pd.read_csv(DATA_DIR / fname, skiprows=1, na_values=["X", "XX"])
    info = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in info]
    d["season_start_year"] = [x[1] for x in info]
    return d

def season_dominant(d, source):
    col = lambda n: d[n] if n in d.columns else 0
    bk = pd.DataFrame({
        "season": d["season"], "season_start_year": d["season_start_year"],
        "A(H1N1)": col("A (H1)") + col("A (2009 H1N1)"),
        "A(H3N2)": col("A (H3)"),
        "B": col("B") + col("BVic") + col("BYam")})
    agg = bk.groupby(["season", "season_start_year"])[["A(H1N1)", "A(H3N2)", "B"]].sum().reset_index()
    agg["total_subtyped"] = agg[["A(H1N1)", "A(H3N2)", "B"]].sum(axis=1)
    agg["dominant_strain"] = agg[["A(H1N1)", "A(H3N2)", "B"]].idxmax(axis=1)
    agg["source"] = source
    return agg

comb = season_dominant(load_nrevss(NREVSS_COMBINED), "Combined")
phl  = season_dominant(load_nrevss(NREVSS_PHL), "PublicHealthLabs")
strain = (pd.concat([comb[comb["season_start_year"] <= 2014],
                     phl[phl["season_start_year"] >= 2015]])
            .sort_values("season_start_year").reset_index(drop=True))
assert strain["season"].is_unique, "stitch overlap: a season got two sources"  # Assertion D

print(strain[["season", "dominant_strain", "A(H1N1)", "A(H3N2)", "B", "total_subtyped", "source"]].to_string(index=False))
print("\nseam check:")
print(strain[strain["season"].isin(["2014-15", "2015-16", "2016-17"])]
      [["season", "source", "A(H1N1)", "A(H3N2)", "B", "dominant_strain"]].to_string(index=False))

## 5. Cleaned per-season output

Targets + flags + dominant strain, one row per complete season. The three flag columns
(`holiday_shift`, `peak_week_smoothing_sensitive`, `fragile_peak_week`) persist so that
`05_forecasting.ipynb` can report peak-week accuracy both including and excluding fragile-label
seasons (the template's +/-1-week metric is partly graded against plateau label noise; we want to
show the result holds on sharply-defined peaks too). Nothing is written to disk here.

In [ ]:
season_table = targets.merge(
    strain[["season", "dominant_strain", "total_subtyped", "source"]], on="season", how="left")
season_table = season_table[
    ["season", "peak_week", "peak_ili_pct", "peak_week_raw", "peak_ili_pct_raw",
     "holiday_shift", "peak_week_smoothing_sensitive", "fragile_peak_week",
     "is_pandemic", "special_case", "dominant_strain", "total_subtyped", "source"]
].sort_values("season").reset_index(drop=True)
print(season_table.to_string(index=False))

print("\nSummary:")
print("  complete seasons:", len(season_table))
print("  holiday_shift:", season_table.loc[season_table["holiday_shift"], "season"].tolist())
print("  smoothing_sensitive:", season_table.loc[season_table["peak_week_smoothing_sensitive"], "season"].tolist())
print("  fragile_peak_week:", season_table.loc[season_table["fragile_peak_week"], "season"].tolist())
print("  special cases:", {r.season: r.special_case for r in season_table.itertuples() if r.special_case})

## Next step

Targets approved on this definition. `03_eda.ipynb` profiles season trajectories, peak-week and
peak-ILI distributions, and missingness, with pandemic / pandemic-adjacent and fragile-label
seasons marked.